# سبق 01 - AI ایجنٹس کا تعارف

**AI ایجنٹس برائے مبتدیوں** کورس کے پہلے سبق میں خوش آمدید!

ایک **AI ایجنٹ** ایک ایسا پروگرام ہے جو ایک بڑی زبان کے ماڈل (LLM) کو اپنے استدلال کے انجن کے طور پر استعمال کرتا ہے اور حقیقی دنیا میں *عمل* کر سکتا ہے — APIs کال کرنا، ڈیٹا بیس سے سوالات کرنا، یا کوڈ چلانا — تاکہ صارف کی جانب سے مقصد حاصل کیا جا سکے۔

اس نوٹ بک میں آپ اپنا پہلا ایجنٹ بنائیں گے: ایک **ٹریول ایجنٹ** جو تعطیلات کی منزلوں کی سفارش کرتا ہے۔ اس دوران آپ یہ سیکھیں گے کہ:

1. **Microsoft Agent Framework** کا استعمال کرتے ہوئے Microsoft Foundry Agent Service سے کیسے جڑا جائے۔
2. ایجنٹ کو ایک **ٹول** دیں — ایک سادہ Python فنکشن جسے وہ کال کر سکتا ہے۔
3. ایجنٹ کو چلائیں اور اس کے جواب کا جائزہ لیں۔
4. ایجنٹ کے جواب کو ٹوکن بہ ٹوکن اسٹریم کریں۔


## سیٹ اپ

اس نوٹ بک کو چلانے سے پہلے، یقینی بنائیں کہ آپ کے پاس درج ذیل ہے:

1. **ایک Microsoft Foundry پروجیکٹ** جس میں ایک تعینات چیٹ ماڈل موجود ہو (مثلاً `gpt-5-mini`)۔
2. **Azure CLI میں لاگ ان ہو چکے ہیں** — اپنے ٹرمینل میں `az login` چلائیں۔
3. **ضروری ماحول کی متغیرات مقرر کریں:**
   - `AZURE_AI_PROJECT_ENDPOINT` — آپ کا Microsoft Foundry پروجیکٹ اینڈپوائنٹ۔
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — آپ کے تعینات شدہ ماڈل کا نام۔

نیچے والا سیل وہ پائتھون پیکجز انسٹال کرے گا جو آپ کو درکار ہیں۔


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## اپنا پہلا ایجنٹ بنانا

ایک ایجنٹ کو دو چیزوں کی ضرورت ہوتی ہے:

- **ہدایات** جو اسے بتاتی ہیں *وہ کون ہے* اور *کیسا رویہ رکھنا ہے* (ایک سسٹم پرامپٹ)۔
- **اوزار** — پائتھن فنکشنز جنہیں `@tool` کے ساتھ ڈیکوریٹ کیا گیا ہے جنہیں ایجنٹ معلومات حاصل کرنے یا کارروائیاں انجام دینے کے لیے کال کر سکتا ہے۔

نیچے ہم ایک سادہ ٹول کی تعریف کرتے ہیں جو مقبول تعطیلاتی مقامات کی فہرست واپس کرتا ہے۔ جب صارف سفر کی تجاویز مانگے گا تو ایجنٹ یہ ٹول استعمال کرے گا۔


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## اسٹریمنگ جوابات

ایک زیادہ باہمی تجربے کے لیے آپ ایجنٹ کے جواب کو **اسٹریمنگ** کر سکتے ہیں۔ مکمل جواب کا انتظار کرنے کے بجائے، ایجنٹ جیسا ہی متن کے ٹکڑے بناتا ہے ویسے ہی انہیں پیش کرتا ہے۔ یہ خاص طور پر چیٹ انٹرفیسز میں مفید ہے جہاں آپ حقیقی وقت میں نتیجہ دکھانا چاہتے ہیں۔


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

- **ایک فراہم کنندہ بنائیں** جو `FoundryChatClient` کے ذریعے Microsoft Foundry Agent Service سے جڑتا ہے۔
- **ایک ٹول کی وضاحت کریں** `@tool` ڈیکوریٹر کا استعمال کرتے ہوئے تاکہ ایجنٹ آپ کے پائتھون فنکشنز کو کال کر سکے۔
- **ایجنٹ چلائیں** صارف کے پیغام کے ساتھ اور اس کا جواب پرنٹ کریں۔
- **جوابات کو اسٹری stream کریں** تاکہ حقیقی وقت میں آؤٹ پٹ ہو۔

اگلے سبق میں ہم ایجنٹک فریم ورک کو مزید تفصیل سے دیکھیں گے اور سیکھیں گے کہ ایجنٹس کو مزید طاقتور ٹولز اور متعدد مراحل کی منطق کی صلاحیتیں کیسے دی جا سکتی ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
